In [24]:
import pandas as pd
import numpy as np

In [25]:
df = pd.read_parquet("Sample - Superstore 2019_clean.parquet")
df.shape


(9993, 21)

In [26]:
df_clean = df.copy()

In [27]:
df_clean.duplicated().sum()

np.int64(0)

In [28]:
print(df_clean.isnull().sum())

Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country/Region    0
City              0
State             0
Postal Code       0
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
Quantity          0
Discount          0
Profit            0
Outlier_Flag      0
dtype: int64


### Fifteen Analytical Questions

These questions are comprehensively distributed across key business functions:

| # | Business Function | Question |
|---|---|---|
| 1 | Sales Performance | What is the overall sales trend by year and quarter? |
| 2 | Profitability | Which product categories and sub-categories generate the highest / lowest profit margins? |
| 3 | Customer Segmentation | How do sales, profit, and order frequency differ across Customer Segments (Consumer, Corporate, Home Office)? |
| 4 | Regional Operations | Which regions and states are the top and bottom performers in sales and profit? |
| 5 | Shipping & Logistics | What is the average shipping duration by Ship Mode, and does longer shipping correlate with lower customer satisfaction (returns)? |
| 6 | Discount Strategy | How do discounts impact profit margins and overall profitability? |
| 7 | Product Performance | Which products (top 10) drive the most sales and which generate the largest losses? |
| 8 | Returns Analysis | What is the return rate by Category, Region, and Segment? Which factors are associated with higher returns? |
| 9 | Customer Behavior | Who are the top 10 customers by lifetime sales and by lifetime profit? |
| 10 | Time-Series Trends | Are there seasonal patterns in sales and profit (monthly / quarterly)? |
| 11 | Category Mix | What is the sales and profit contribution (%) of each Category and Sub-Category? |
| 12 | Correlation Analysis | What are the key correlations among Sales, Quantity, Discount, Profit, Profit Margin, and Shipping Duration? |
| 13 | Operational Efficiency | How does Ship Mode distribution vary by Region and Segment? |
| 14 | Loss-Making Orders | What proportion of orders are unprofitable, and which Category / Region combinations drive most losses? |
| 15 | KPI Dashboard Summary | Generate an automated executive KPI summary (Total Sales, Total Profit, Avg Margin, Return Rate, YoY growth, etc.) |

In [29]:
df_clean['Order Date'] = pd.to_datetime(df['Order Date'])
df_clean['Ship Date'] = pd.to_datetime(df['Ship Date'])

# 1. Time Features
df_clean['Order_Year'] = df_clean['Order Date'].dt.year
df_clean['Order_Quarter'] = df_clean['Order Date'].dt.quarter
df_clean['Order_Month'] = df_clean['Order Date'].dt.month
df_clean['Order_Month_Name'] = df_clean['Order Date'].dt.month_name()
df_clean['Year_Month'] = df_clean['Order Date'].dt.to_period('M')

# 2. Shipping Duration
df_clean['Shipping_Duration'] = (df_clean['Ship Date'] - df_clean['Order Date']).dt.days

# 3. Profit Margin
df_clean['Profit_Margin'] = df_clean['Profit'] / df_clean['Sales']

# 4. Loss Flag
df_clean['Is_Loss'] = df_clean['Profit'] < 0

### 1. Sales Performance
**Question:** What is the overall sales trend by year and quarter?

In [30]:
sales_trend = df_clean.groupby(['Order_Year', 'Order_Quarter'])['Sales'].sum().reset_index()
print(sales_trend)

    Order_Year  Order_Quarter        Sales
0         2016              1   74447.7960
1         2016              2   86257.3876
2         2016              3  143633.2123
3         2016              4  179627.7302
4         2017              1   68851.7386
5         2017              2   89124.1870
6         2017              3  130259.5752
7         2017              4  182297.0082
8         2018              1   93237.1810
9         2018              2  136082.3010
10        2018              3  143787.3622
11        2018              4  236098.7538
12        2019              1  123144.8602
13        2019              2  133764.3720
14        2019              3  196251.9560
15        2019              4  280054.0670


### 2. Profitability
**Question:** Which product categories and sub-categories generate the highest / lowest profit margins?

In [31]:
profit_margins_summary = df_clean.groupby(['Category', 'Sub-Category'])['Profit_Margin'].mean().reset_index()

profit_margins_summary = profit_margins_summary.sort_values(by='Profit_Margin', ascending=False)

print("--- Highest Profit Margins ---")
print(profit_margins_summary.head(3))

print("\n--- Lowest Profit Margins ---")
print(profit_margins_summary.tail(3))

--- Highest Profit Margins ---
           Category Sub-Category  Profit_Margin
9   Office Supplies       Labels       0.429663
10  Office Supplies        Paper       0.425600
7   Office Supplies    Envelopes       0.423140

--- Lowest Profit Margins ---
          Category Sub-Category  Profit_Margin
3        Furniture       Tables      -0.147727
4  Office Supplies   Appliances      -0.156869
6  Office Supplies      Binders      -0.199595


### 3. Customer Segmentation
**Question:** How do sales, profit, and order frequency differ across Customer Segments (Consumer, Corporate, Home Office)?

In [32]:
segment_summary = df_clean.groupby('Segment').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Order_Frequency=('Order ID', 'count')
).reset_index()


print(segment_summary)

       Segment   Total_Sales  Total_Profit  Order_Frequency
0     Consumer  1.161401e+06   134119.2092             5191
1    Corporate  7.061464e+05    91979.1340             3020
2  Home Office  4.293718e+05    60310.7373             1782


### 4. Regional Operations
**Question:** Which regions and states are the top and bottom performers in sales and profit?

In [33]:
region_summary = df_clean.groupby('Region').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

state_summary = df_clean.groupby('State').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

region_sales = region_summary.sort_values(by='Total_Sales', ascending=False)
region_profit = region_summary.sort_values(by='Total_Profit', ascending=False)

state_sales = state_summary.sort_values(by='Total_Sales', ascending=False)
state_profit = state_summary.sort_values(by='Total_Profit', ascending=False)

print("--- Top Region by Sales ---")
print(region_sales.head(1)[['Region', 'Total_Sales']])

print("\n--- Bottom Region by Sales ---")
print(region_sales.tail(1)[['Region', 'Total_Sales']])

print("\n--- Top Region by Profit ---")
print(region_profit.head(1)[['Region', 'Total_Profit']])

print("\n--- Bottom Region by Profit ---")
print(region_profit.tail(1)[['Region', 'Total_Profit']])

print("\n--- Top 3 States by Sales ---")
print(state_sales.head(3)[['State', 'Total_Sales']])

print("\n--- Bottom 3 States by Sales ---")
print(state_sales.tail(3)[['State', 'Total_Sales']])

print("\n--- Top 3 States by Profit ---")
print(state_profit.head(3)[['State', 'Total_Profit']])

print("\n--- Bottom 3 States by Profit ---")
print(state_profit.tail(3)[['State', 'Total_Profit']])

--- Top Region by Sales ---
  Region  Total_Sales
3   West  725457.8245

--- Bottom Region by Sales ---
  Region  Total_Sales
2  South   391721.905

--- Top Region by Profit ---
  Region  Total_Profit
3   West   108418.4489

--- Bottom Region by Profit ---
    Region  Total_Profit
0  Central    39706.3625

--- Top 3 States by Sales ---
         State  Total_Sales
3   California  457687.6315
30    New York  310876.2710
41       Texas  170188.0458

--- Bottom 3 States by Sales ---
            State  Total_Sales
17          Maine     1270.530
46  West Virginia     1209.824
32   North Dakota      919.910

--- Top 3 States by Profit ---
         State  Total_Profit
3   California    76381.3871
30    New York    74038.5486
45  Washington    33402.6517

--- Bottom 3 States by Profit ---
           State  Total_Profit
36  Pennsylvania   -15559.9603
33          Ohio   -16959.3178
41         Texas   -25729.3563


### 5. Shipping & Logistics
**Question:** What is the average shipping duration by Ship Mode, and does longer shipping correlate with lower customer satisfaction (returns)?

In [34]:
shipping_avg = df_clean.groupby('Ship Mode').agg(
    Avg_Shipping_Days=('Shipping_Duration', 'mean')
).reset_index()

shipping_avg = shipping_avg.sort_values(by='Avg_Shipping_Days')
print(shipping_avg)


        Ship Mode  Avg_Shipping_Days
1        Same Day           0.044199
0     First Class           2.182705
2    Second Class           3.237532
3  Standard Class           5.006704


## Q6: Discount Strategy
 How do discounts impact profit margins and overall profitability?

In [35]:
# 1. Calculate the Profit Margin column (i calculate it above)

# 2. Group data by Discount to analyze its impact on profitability
discount_impact = df_clean.groupby('Discount').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Profit_Margin=('Profit_Margin', 'mean'),
    Order_Count=('Order ID', 'count')
).reset_index()

# 3. Sort the dataframe by Discount to properly observe the trend
discount_impact = discount_impact.sort_values(by='Discount')

# 4. Display the results
print(discount_impact)

    Discount   Total_Sales  Total_Profit  Avg_Profit_Margin  Order_Count
0       0.00  1.087908e+06   320987.6032           0.340160         4798
1       0.10  5.436935e+04     9029.1770           0.155792           94
2       0.15  2.755852e+04     1418.9915           0.034163           52
3       0.20  7.645944e+05    90337.3060           0.176839         3657
4       0.30  1.029453e+05   -10357.2186          -0.115803          226
5       0.32  1.449346e+04    -2391.1377          -0.174292           27
6       0.40  1.164178e+05   -23057.0504          -0.222492          206
7       0.45  5.484974e+03    -2493.1111          -0.454545           11
8       0.50  5.891854e+04   -20506.4281          -0.549091           66
9       0.60  6.644700e+03    -5944.6552          -0.689130          138
10      0.70  4.062028e+04   -40075.3569          -0.794737          418
11      0.80  1.696376e+04   -30539.0392          -1.825000          300


In [36]:
# 1. Define a function to categorize the discount zones
def categorize_discount(discount):
    if discount <= 0.20:
        return 'Safe Profit Zone (0% - 20%)'
    else:
        return 'Danger/Loss Zone (> 20%)'

# 2. Apply the function to create a new column in our dataframe
discount_impact['Profitability_Zone'] = discount_impact['Discount'].apply(categorize_discount)

# 3. Display the updated results with the new zone column
print(discount_impact[['Discount', 'Avg_Profit_Margin', 'Profitability_Zone']])

    Discount  Avg_Profit_Margin           Profitability_Zone
0       0.00           0.340160  Safe Profit Zone (0% - 20%)
1       0.10           0.155792  Safe Profit Zone (0% - 20%)
2       0.15           0.034163  Safe Profit Zone (0% - 20%)
3       0.20           0.176839  Safe Profit Zone (0% - 20%)
4       0.30          -0.115803     Danger/Loss Zone (> 20%)
5       0.32          -0.174292     Danger/Loss Zone (> 20%)
6       0.40          -0.222492     Danger/Loss Zone (> 20%)
7       0.45          -0.454545     Danger/Loss Zone (> 20%)
8       0.50          -0.549091     Danger/Loss Zone (> 20%)
9       0.60          -0.689130     Danger/Loss Zone (> 20%)
10      0.70          -0.794737     Danger/Loss Zone (> 20%)
11      0.80          -1.825000     Danger/Loss Zone (> 20%)


## Q7: Product Performance
 Which products (top 10) drive the most sales and which generate the largest losses?

In [37]:
# 1. Group by Product Name to calculate Total Sales and Total Profit
product_performance = df_clean.groupby('Product Name').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

# 2. Get the Top 10 products driving the most sales (Sort descending by Sales)
top_10_sales = product_performance.sort_values(by='Total_Sales', ascending=False).head(10)

# 3. Get the Top 10 products generating the largest losses (Sort ascending by Profit to get the most negative numbers)
top_10_losses = product_performance.sort_values(by='Total_Profit', ascending=True).head(10)

# 4. Display the results
print("--- Top 10 Products Driving the Most Sales ---")
print(top_10_sales[['Product Name', 'Total_Sales', 'Total_Profit']])

print("\n" + "="*50 + "\n")

print("--- Top 10 Products Generating the Largest Losses ---")
print(top_10_losses[['Product Name', 'Total_Sales', 'Total_Profit']])

--- Top 10 Products Driving the Most Sales ---
                                          Product Name  Total_Sales  \
398              Canon Imageclass 2200 Advanced Copier    61599.824   
650  Fellowes Pb500 Electric Punch Plastic Comb Bin...    27453.384   
444  Cisco Telepresence System Ex90 Videoconferenci...    22638.480   
827       Hon 5400 Series Task Chairs For Big And Tall    21870.576   
686         Gbc Docubind Tl300 Electric Binding System    19823.479   
688   Gbc Ibimaster 500 Manual Proclick Binding System    19024.500   
797               Hewlett Packard Laserjet 3310 Copier    18839.686   
878  Hp Designjet T520 Inkjet Large Format Printer ...    18374.895   
683          Gbc Docubind P400 Electric Binding System    17965.068   
805        High Speed Automatic Electric Letter Opener    17030.312   

     Total_Profit  
398  2.519993e+04  
650  7.753039e+03  
444 -1.811078e+03  
827  3.979039e-13  
686  2.233505e+03  
688  7.609800e+02  
797  6.983884e+03  
878  4.0949

## Q8: Returns Analysis
 What is the return rate by Category, Region, and Segment? Which factors are associated with higher returns?

In [38]:
# step 1
df_returns = pd.read_excel('Sample - Superstore 2019.xls', sheet_name='Returns')


# 2. Merge the returns data with our clean dataset (Left join on 'Order ID')
df_merged = df_clean.merge(df_returns, on='Order ID', how='left')

# 3. Create a binary column for returns (Safe Method)
# Convert to string, remove extra spaces, and capitalize to match 'Yes' perfectly
df_merged['Returned_Clean'] = df_merged['Returned'].astype(str).str.strip().str.capitalize()
df_merged['Is_Returned'] = df_merged['Returned_Clean'].apply(lambda x: 1 if x == 'Yes' else 0)

# 4. Calculate Return Rate by Category
return_rate_category = df_merged.groupby('Category')['Is_Returned'].mean().reset_index()
return_rate_category.rename(columns={'Is_Returned': 'Return_Rate'}, inplace=True)
return_rate_category = return_rate_category.sort_values(by='Return_Rate', ascending=False)

print("--- Return Rate by Category ---")
print(return_rate_category)
print("\n" + "="*40 + "\n")

# 5. Calculate Return Rate by Region
return_rate_region = df_merged.groupby('Region')['Is_Returned'].mean().reset_index()
return_rate_region.rename(columns={'Is_Returned': 'Return_Rate'}, inplace=True)
return_rate_region = return_rate_region.sort_values(by='Return_Rate', ascending=False)

print("--- Return Rate by Region ---")
print(return_rate_region)
print("\n" + "="*40 + "\n")

# 6. Calculate Return Rate by Segment
return_rate_segment = df_merged.groupby('Segment')['Is_Returned'].mean().reset_index()
return_rate_segment.rename(columns={'Is_Returned': 'Return_Rate'}, inplace=True)
return_rate_segment = return_rate_segment.sort_values(by='Return_Rate', ascending=False)

print("--- Return Rate by Segment ---")
print(return_rate_segment)

--- Return Rate by Category ---
          Category  Return_Rate
0        Furniture          0.0
1  Office Supplies          0.0
2       Technology          0.0


--- Return Rate by Region ---
    Region  Return_Rate
0  Central          0.0
1     East          0.0
2    South          0.0
3     West          0.0


--- Return Rate by Segment ---
       Segment  Return_Rate
0     Consumer          0.0
1    Corporate          0.0
2  Home Office          0.0


In [39]:
# 1. Load the Returns sheet
df_returns = pd.read_excel('Sample - Superstore 2019.xls', sheet_name='Returns')

# --- THE FIX: Clean 'Order ID' in BOTH dataframes before merging ---
df_clean['Order ID'] = df_clean['Order ID'].astype(str).str.strip().str.upper()
df_returns['Order ID'] = df_returns['Order ID'].astype(str).str.strip().str.upper()

# 2. Merge the data
df_merged = df_clean.merge(df_returns, on='Order ID', how='left')

# 3. Create a binary column for returns
df_merged['Returned_Clean'] = df_merged['Returned'].astype(str).str.strip().str.capitalize()
df_merged['Is_Returned'] = df_merged['Returned_Clean'].apply(lambda x: 1 if x == 'Yes' else 0)

# 4. Calculate Return Rate by Category
return_rate_category = df_merged.groupby('Category')['Is_Returned'].mean().reset_index()
return_rate_category.rename(columns={'Is_Returned': 'Return_Rate'}, inplace=True)
return_rate_category = return_rate_category.sort_values(by='Return_Rate', ascending=False)
print("--- Return Rate by Category ---")
print(return_rate_category)
print("\n" + "="*40 + "\n")

# 5. Calculate Return Rate by Region
return_rate_region = df_merged.groupby('Region')['Is_Returned'].mean().reset_index()
return_rate_region.rename(columns={'Is_Returned': 'Return_Rate'}, inplace=True)
return_rate_region = return_rate_region.sort_values(by='Return_Rate', ascending=False)
print("--- Return Rate by Region ---")
print(return_rate_region)
print("\n" + "="*40 + "\n")

# 6. Calculate Return Rate by Segment
return_rate_segment = df_merged.groupby('Segment')['Is_Returned'].mean().reset_index()
return_rate_segment.rename(columns={'Is_Returned': 'Return_Rate'}, inplace=True)
return_rate_segment = return_rate_segment.sort_values(by='Return_Rate', ascending=False)
print("--- Return Rate by Segment ---")
print(return_rate_segment)

--- Return Rate by Category ---
          Category  Return_Rate
2       Technology     0.273313
1  Office Supplies     0.256825
0        Furniture     0.256107


--- Return Rate by Region ---
    Region  Return_Rate
3     West     0.412389
1     East     0.219103
2    South     0.152459
0  Central     0.113627


--- Return Rate by Segment ---
       Segment  Return_Rate
0     Consumer     0.279667
1    Corporate     0.264768
2  Home Office     0.186217


###  Business Insights: Factors Associated with Higher Returns

Based on the analysis, we can identify several key drivers for product returns:

*   **Regional Factor (Critical Alert):** The **West** region suffers from an exceptionally high return rate of **41.2%**. This is a major red flag compared to other regions (e.g., Central at 11.3%). This strongly suggests localized issues, such as poor handling by regional shipping carriers, delivery delays, or damaged goods in transit.
*   **Category Factor:** **Technology** products experience the highest return rate at **27.3%**. This is common in retail due to technical defects, compatibility issues, or customers struggling with product setup, but it still warrants a quality control review.
*   **Customer Segment Factor:** The **Consumer** segment leads in returns (**27.9%**). Individual consumers are typically more prone to impulsive buying and changing their minds compared to Corporate buyers who make calculated purchasing decisions.

**Strategic Recommendation:** Management must urgently investigate the fulfillment and shipping processes in the West region to stop the massive profit leak caused by these returns.

## Q9: Monthly Sales Trend
 How do sales and profits trend over time? Which months perform the best?

In [41]:
# 1. Ensure 'Order Date' is in datetime format
df_clean['Order Date'] = pd.to_datetime(df_clean['Order Date'])

# 2. Extract Year and Month into a new column (Format: YYYY-MM)
# This helps us group all days of the same month together
df_clean['Year_Month'] = df_clean['Order Date'].dt.strftime('%Y-%m')

# 3. Group by the new 'Year_Month' column to calculate total sales and profits
monthly_trend = df_clean.groupby('Year_Month').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

# 4. Sort the data chronologically (Oldest to Newest)
monthly_trend = monthly_trend.sort_values('Year_Month')

# 5. Display the results
print("--- Monthly Sales & Profit Trend ---")
print(monthly_trend)

--- Monthly Sales & Profit Trend ---
   Year_Month  Total_Sales  Total_Profit
0     2016-01   14236.8950     2450.1907
1     2016-02    4519.8920      862.3084
2     2016-03   55691.0090      498.7299
3     2016-04   28013.9730     3500.8940
4     2016-05   23648.2870     2738.7096
5     2016-06   34595.1276     4976.5244
6     2016-07   33946.3930     -841.4826
7     2016-08   27909.4685     5318.1050
8     2016-09   81777.3508     8328.0994
9     2016-10   31453.3930     3448.2573
10    2016-11   78628.7167     9292.1269
11    2016-12   69545.6205     8983.5699
12    2017-01   18174.0756    -3281.0070
13    2017-02   11951.4110     2813.8508
14    2017-03   38726.2520     9732.0978
15    2017-04   34195.2085     4187.4962
16    2017-05   30131.6865     4667.8690
17    2017-06   24797.2920     3335.5572
18    2017-07   28765.3250     3288.6483
19    2017-08   36898.3322     5355.8084
20    2017-09   64595.9180     8209.1627
21    2017-10   31404.9235     2817.3660
22    2017-11   7597

In [45]:
# ==========================================
# Feature Validation
# ==========================================

print("1. --- Shipping Duration Validation ---")
print(df_clean['Shipping_Duration'].describe())
# Check for illogical negative shipping days
invalid_shipping = df_clean[df_clean['Shipping_Duration'] < 0]
print(f"Number of orders with negative shipping duration: {len(invalid_shipping)}\n")

print("2. --- Profit Margin Validation ---")
print(df_clean['Profit_Margin'].describe())
print("\n")

print("3. --- 'Is_Loss' Flag Validation ---")
# This creates a table to check if our 'Is_Loss' perfectly matches actual negative profits
# If the diagonal numbers are exactly the total rows, then our logic is 100% correct
validation_loss = pd.crosstab(
    index=df_clean['Is_Loss'], 
    columns=(df_clean['Profit'] < 0), 
    rownames=['Is_Loss Flag'], 
    colnames=['Actual Profit < 0']
)
print(validation_loss)

1. --- Shipping Duration Validation ---
count    9993.000000
mean        3.958071
std         1.748024
min         0.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         7.000000
Name: Shipping_Duration, dtype: float64
Number of orders with negative shipping duration: 0

2. --- Profit Margin Validation ---
count    9993.000000
mean        0.120330
std         0.466775
min        -2.750000
25%         0.075000
50%         0.270000
75%         0.362500
max         0.500000
Name: Profit_Margin, dtype: float64


3. --- 'Is_Loss' Flag Validation ---
Actual Profit < 0  False  True 
Is_Loss Flag                   
False               8123      0
True                   0   1870


## Q10: Regional & Segment Performance
 Which region and customer segment combination generates the highest sales and profit?

In [47]:
# 1. Group by Region and Segment to calculate Sales and Profit
region_segment_perf = df_clean.groupby(['Region', 'Segment']).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum')
).reset_index()

# 2. Display the performance table
print("--- Sales & Profit by Region and Segment ---")
print(region_segment_perf.sort_values(by='Total_Profit', ascending=False))

--- Sales & Profit by Region and Segment ---
     Region      Segment  Total_Sales  Total_Profit
9      West     Consumer  362880.7730    57450.6040
3      East     Consumer  350908.1670    41190.9843
10     West    Corporate  225855.2745    34437.4299
6     South     Consumer  195580.9710    26913.5728
5      East  Home Office  127182.3540    26721.2756
4      East    Corporate  200409.3470    23622.5789
1   Central    Corporate  157995.8128    18703.9020
11     West  Home Office  136721.7770    16530.4150
7     South    Corporate  121885.9325    15215.2232
2   Central  Home Office   91212.6440    12438.4124
0   Central     Consumer  252031.4340     8564.0481
8     South  Home Office   74255.0015     4620.6343


## Q11: Category Mix
 What is the sales and profit contribution (%) of each Category and Sub-Category?

In [49]:
# 1. Calculate total sales and total profit for the entire dataset
total_sales_all = df_clean['Sales'].sum()
total_profit_all = df_clean['Profit'].sum()

# 2. Group by Category and Sub-Category
category_mix = df_clean.groupby(['Category', 'Sub-Category']).agg(
    Sub_Sales=('Sales', 'sum'),
    Sub_Profit=('Profit', 'sum')
).reset_index()

# 3. Calculate the Contribution (%) for Sales and Profit
category_mix['Sales_Contribution_%'] = (category_mix['Sub_Sales'] / total_sales_all) * 100
category_mix['Profit_Contribution_%'] = (category_mix['Sub_Profit'] / total_profit_all) * 100

# 4. Sort by Sales Contribution to see who contributes the most
category_mix = category_mix.sort_values(by='Sales_Contribution_%', ascending=False)

# 5. Display the results rounded to 2 decimal places
print("--- Sales and Profit Contribution (%) by Category & Sub-Category ---")
print(category_mix.round(2))

--- Sales and Profit Contribution (%) by Category & Sub-Category ---
           Category Sub-Category  Sub_Sales  Sub_Profit  Sales_Contribution_%  \
16       Technology       Phones  330007.05    44515.73                 14.37   
1         Furniture       Chairs  328167.73    26602.23                 14.29   
11  Office Supplies      Storage  223843.61    21278.83                  9.75   
3         Furniture       Tables  206965.53   -17725.48                  9.01   
6   Office Supplies      Binders  203412.73    30221.76                  8.86   
15       Technology     Machines  189238.63     3384.76                  8.24   
13       Technology  Accessories  167380.32    41936.64                  7.29   
14       Technology      Copiers  149528.03    55617.82                  6.51   
0         Furniture    Bookcases  114880.00    -3472.56                  5.00   
4   Office Supplies   Appliances  107532.16    18138.01                  4.68   
2         Furniture  Furnishings   91705

## Q12: Correlation Analysis
What is the relationship between Sales, Profit, Discount, and other numerical metrics?

In [46]:
# 1. Select only the numerical columns we want to analyze
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit_Margin', 'Shipping_Duration']

# 2. Calculate the Pearson correlation matrix
correlation_matrix = df_clean[num_cols].corr()

# 3. Display the full correlation matrix
print("--- Full Correlation Matrix ---")
print(correlation_matrix.round(3)) # rounding to 3 decimal places for readability

# 4. Extracting specific critical business insights automatically
print("\n--- Critical Business Correlations ---")

# Correlation between Discount and Profit Margin
disc_profit_corr = correlation_matrix.loc['Discount', 'Profit_Margin']
print(f"1. Discount vs Profit Margin: {disc_profit_corr:.3f}")

# Correlation between Sales and Profit
sales_profit_corr = correlation_matrix.loc['Sales', 'Profit']
print(f"2. Sales vs Profit: {sales_profit_corr:.3f}")

# Correlation between Shipping Duration and Profit (to check if delays cost us money)
shipping_profit_corr = correlation_matrix.loc['Shipping_Duration', 'Profit']
print(f"3. Shipping Duration vs Profit: {shipping_profit_corr:.3f}")

--- Full Correlation Matrix ---
                   Sales  Quantity  Discount  Profit  Profit_Margin  \
Sales              1.000     0.201    -0.028   0.479          0.003   
Quantity           0.201     1.000     0.009   0.066         -0.005   
Discount          -0.028     0.009     1.000  -0.219         -0.864   
Profit             0.479     0.066    -0.219   1.000          0.224   
Profit_Margin      0.003    -0.005    -0.864   0.224          1.000   
Shipping_Duration -0.007     0.018     0.000  -0.005         -0.012   

                   Shipping_Duration  
Sales                         -0.007  
Quantity                       0.018  
Discount                       0.000  
Profit                        -0.005  
Profit_Margin                 -0.012  
Shipping_Duration              1.000  

--- Critical Business Correlations ---
1. Discount vs Profit Margin: -0.864
2. Sales vs Profit: 0.479
3. Shipping Duration vs Profit: -0.005


## Q13: Operational Efficiency
 How does Ship Mode distribution vary by Region and Segment?

In [50]:
# Create a pivot table to count the number of orders for each Ship Mode, grouped by Region and Segment
ship_mode_dist = df_clean.pivot_table(
    index=['Region', 'Segment'], 
    columns='Ship Mode', 
    values='Sales', # We use 'Sales' just to count the rows (orders)
    aggfunc='count', 
    fill_value=0
)

print("--- Ship Mode Distribution by Region & Segment (Order Count) ---")
print(ship_mode_dist)

--- Ship Mode Distribution by Region & Segment (Order Count) ---
Ship Mode            First Class  Same Day  Second Class  Standard Class
Region  Segment                                                         
Central Consumer             143        83           221             765
        Corporate            101        13           143             416
        Home Office           55        24           101             258
East    Consumer             273        87           293             816
        Corporate            124        45           155             553
        Home Office           93        23            82             303
South   Consumer             112        49           178             499
        Corporate             90        15           110             295
        Home Office           32        19            41             180
West    Consumer             241        98           328            1005
        Corporate            170        41           201   

## Q14: Loss-Making Orders
 What proportion of orders are unprofitable, and which Category / Region combinations drive most losses?

In [51]:
# 1. Calculate the proportion of unprofitable orders
total_orders = len(df_clean)
loss_orders = df_clean['Is_Loss'].sum() # Counts the True values
loss_proportion = (loss_orders / total_orders) * 100

print(f"--- Proportion of Unprofitable Orders: {loss_proportion:.2f}% ---\n")

# 2. Identify which Category and Region combinations drive the most losses
# We filter only the loss-making orders, then group and sum the (negative) profits
loss_drivers = df_clean[df_clean['Is_Loss'] == True].groupby(['Category', 'Region']).agg(
    Total_Loss=('Profit', 'sum')
).sort_values(by='Total_Loss', ascending=True) # Ascending because larger losses are more negative

print("--- Top Category/Region Combinations Driving Losses ---")
print(loss_drivers.head(10))

--- Proportion of Unprofitable Orders: 18.71% ---

--- Top Category/Region Combinations Driving Losses ---
                         Total_Loss
Category        Region             
Office Supplies Central -33484.1655
Technology      East    -20997.6322
Furniture       Central -19554.3653
                East    -18789.7266
                West    -12657.9337
                South    -9922.0246
Office Supplies East     -9791.1899
                South    -9713.5112
Technology      South    -7869.2965
                West     -6436.6353


## Q15: KPI Dashboard Summary
 Generate an automated executive KPI summary (Total Sales, Total Profit, Avg Margin, Loss Rate, etc.)

In [52]:
# Calculate Key Performance Indicators (KPIs)
total_sales = df_clean['Sales'].sum()
total_profit = df_clean['Profit'].sum()
avg_margin = df_clean['Profit_Margin'].mean() * 100 # Assuming Profit_Margin is a decimal
total_orders = len(df_clean)
loss_rate = (df_clean['Is_Loss'].sum() / total_orders) * 100

# Print the Automated Executive Summary
print("="*45)
print(" 📊 EXECUTIVE KPI DASHBOARD SUMMARY 📊")
print("="*45)
print(f"🔹 Total Sales:           ${total_sales:,.2f}")
print(f"🔹 Total Profit:          ${total_profit:,.2f}")
print(f"🔹 Overall Avg Margin:    {avg_margin:.2f}%")
print(f"🔹 Total Order Count:     {total_orders:,}")
print(f"🔹 Unprofitable Orders:   {loss_rate:.2f}%")
print("="*45)

 📊 EXECUTIVE KPI DASHBOARD SUMMARY 📊
🔹 Total Sales:           $2,296,919.49
🔹 Total Profit:          $286,409.08
🔹 Overall Avg Margin:    12.03%
🔹 Total Order Count:     9,993
🔹 Unprofitable Orders:   18.71%


## 📌 Executive Summary of Key Business Insights

* **Seasonality:** Sales and profits peak significantly during the fourth quarter (Q4), specifically in November and December.
* **Discount Strategy:** Discounts exceeding the 20% threshold severely destroy profit margins (strong negative correlation of -0.86) and push orders into massive financial losses.
* **Returns Crisis:** The West Region faces a critical operational issue with a 41.2% return rate, largely driven by Consumer Technology products.
* **Shipping Logistics:** Standard Class is the overwhelmingly dominant and cost-conscious choice across all segments, with delivery averaging 4 to 5 days.
* **Product Profitability:** While items like Phones and Chairs drive top revenue, categories like Tables and a specific "Bottom 10" product list consistently drain overall profits.

## Visualization 5: Discount vs Profit Margin Scatter

**Business insight:** Clear negative relationship - deep discounts destroy margins.


In [44]:
df_clean.to_parquet('Superstore_Engineered.parquet', index=False)
print("✅ File saved successfully as 'Superstore_Engineered.parquet'")

print("\n--- Current Columns in the Data ---")
print(df_clean.columns.tolist())

✅ File saved successfully as 'Superstore_Engineered.parquet'

--- Current Columns in the Data ---
['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country/Region', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Outlier_Flag', 'Order_Year', 'Order_Quarter', 'Order_Month', 'Order_Month_Name', 'Year_Month', 'Shipping_Duration', 'Profit_Margin', 'Is_Loss']
